In [ ]:
# ─── 1. Instalar dependências ────────────────────────────────────────────────
!pip install openai -q
print('✅ Dependências instaladas!')

In [ ]:
# ─── 2. Configuração ────────────────────────────────────────────────────────
import os
from openai import OpenAI
from google.colab import userdata

# Adicione OPENAI_API_KEY nos Secrets do Colab (🔑 ícone no menu lateral)
try:
    api_key = userdata.get('OPENAI_API_KEY')
except:
    api_key = input('Cole sua OpenAI API Key: ')

client = OpenAI(api_key=api_key)

SYSTEM_PROMPT = """
Você é a FIN, uma assistente financeira virtual inteligente, empática e didática.
Ajude usuários brasileiros a entender suas finanças pessoais com linguagem clara.

CAPACIDADES:
- Explicar produtos financeiros (CDB, Tesouro Direto, fundos, poupança)
- Realizar e explicar cálculos financeiros com exemplos em reais (R$)
- Dar dicas de planejamento e controle financeiro
- Responder FAQs sobre bancos, cartões e investimentos

REGRAS:
- Sempre em português brasileiro
- Use emojis moderadamente
- Nunca garanta retornos específicos
- Sempre sugira consultar profissional certificado para decisões importantes
"""

print('✅ Configuração concluída!')

In [ ]:
# ─── 3. Funções de Cálculo Financeiro ────────────────────────────────────────

def calcular_juros_compostos(principal, taxa_mensal, meses):
    montante = principal * ((1 + taxa_mensal/100) ** meses)
    juros = montante - principal
    print(f"\n📊 Simulação — Juros Compostos")
    print(f"{'─'*40}")
    print(f"  Capital inicial:   R$ {principal:>10,.2f}")
    print(f"  Taxa mensal:            {taxa_mensal:>6}%")
    print(f"  Período:               {meses:>6} meses")
    print(f"{'─'*40}")
    print(f"  Montante final:    R$ {montante:>10,.2f}")
    print(f"  Juros ganhos:      R$ {juros:>10,.2f}")
    print(f"  Rendimento:              {(juros/principal*100):>5.1f}%")
    print(f"{'─'*40}")
    return montante

def calcular_financiamento(valor, taxa_mensal, parcelas):
    t = taxa_mensal / 100
    p = valor * (t * (1+t)**parcelas) / ((1+t)**parcelas - 1)
    total = p * parcelas
    juros = total - valor
    print(f"\n🏦 Simulação — Financiamento (Tabela Price)")
    print(f"{'─'*40}")
    print(f"  Valor financiado:  R$ {valor:>10,.2f}")
    print(f"  Taxa mensal:            {taxa_mensal:>6}%")
    print(f"  Parcelas:              {parcelas:>6}x")
    print(f"{'─'*40}")
    print(f"  Valor da parcela:  R$ {p:>10,.2f}")
    print(f"  Total a pagar:     R$ {total:>10,.2f}")
    print(f"  Total de juros:    R$ {juros:>10,.2f}")
    print(f"{'─'*40}")
    return p

def calcular_reserva_emergencia(renda, meses=6):
    reserva = renda * meses
    print(f"\n🛡️  Reserva de Emergência Ideal")
    print(f"{'─'*40}")
    print(f"  Renda mensal:      R$ {renda:>10,.2f}")
    print(f"  Meses recomendados:     {meses:>6}")
    print(f"{'─'*40}")
    print(f"  Reserva ideal:     R$ {reserva:>10,.2f}")
    print(f"  Para atingir em 6m:R$ {reserva/6:>10,.2f}/mês")
    print(f"  Para atingir em 1a:R$ {reserva/12:>10,.2f}/mês")
    print(f"{'─'*40}")
    return reserva

print('✅ Funções de cálculo prontas!')

In [ ]:
# ─── 4. Assistente FIN com contexto persistente ──────────────────────────────

historico = []

def fin(pergunta: str) -> str:
    historico.append({"role": "user", "content": pergunta})
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role":"system","content":SYSTEM_PROMPT}] + historico[-10:],
        max_tokens=800,
        temperature=0.7
    )
    resposta = response.choices[0].message.content.strip()
    historico.append({"role": "assistant", "content": resposta})
    print(f"\n🤖 FIN: {resposta}\n")
    return resposta

print('✅ FIN está pronta para conversar!')

In [ ]:
# ─── 5. DEMONSTRAÇÃO — Perguntas para a FIN ─────────────────────────────────
print('='*55)
print('  💰  FIN — Assistente Financeira Virtual')
print('='*55)

fin('Olá! O que você pode fazer por mim?')

In [ ]:
fin('Qual a diferença entre CDB e Tesouro Direto?')

In [ ]:
# ─── 6. Simulação Financeira ─────────────────────────────────────────────────
# Juros compostos: R$10.000 a 1% ao mês por 36 meses
calcular_juros_compostos(10000, 1, 36)

In [ ]:
# Financiamento: R$50.000 a 1.5% ao mês em 60 parcelas
calcular_financiamento(50000, 1.5, 60)

In [ ]:
# Reserva de emergência para renda de R$4.000/mês
calcular_reserva_emergencia(4000)

In [ ]:
# ─── 7. Chat Interativo ──────────────────────────────────────────────────────
print('💬 Digite sua pergunta (ou "sair" para encerrar):\n')
while True:
    p = input('Você: ').strip()
    if p.lower() in ['sair','exit','quit']:
        print('👋 FIN: Até logo! Boas finanças! 💚')
        break
    if p:
        fin(p)